## Product Intelligence

### Objetivo

Analizar cómo cambia la lectura del rendimiento de producto cuando se incorporan las devoluciones.

La pregunta principal es:

**¿Los productos que destacan por demanda bruta mantienen su rendimiento cuando se analiza lo que el cliente realmente conserva?**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [39]:
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

fact_transactions.to_parquet(
    PROCESSED_PATH / "fact_transactions.parquet",
    index=False
)

fact_orders.to_parquet(
    PROCESSED_PATH / "fact_orders.parquet",
    index=False
)

dim_product.to_parquet(
    PROCESSED_PATH / "dim_product.parquet",
    index=False
)

dim_date.to_parquet(
    PROCESSED_PATH / "dim_date.parquet",
    index=False
)

In [3]:
fact_transactions = df.loc[
    df["is_valid_transaction"]
].copy()

In [4]:
fact_transactions = fact_transactions[
    [
        "orderID",
        "orderDate",
        "articleID",
        "customerID",
        "colorCode",
        "sizeCode",
        "productGroup",

        "ordered_units",
        "returned_units",
        "kept_units",

        "ordered_sales_value",
        "returned_sales_value",
        "kept_sales_value",

        "rrp",
        "unit_selling_price",
        "markdown_amount",
        "markdown_pct",
        "price_status",

        "voucherAmount",
        "has_voucher",

        "is_zero_value_item",
        "has_missing_product_metadata"
    ]
].copy()

In [5]:
fact_transactions.shape

(2293521, 22)

In [21]:
dim_product = (
    fact_transactions
    .groupby("articleID")
    .agg(
        productGroup=("productGroup", "first"),
        rrp=("rrp", "first")
    )
    .reset_index()
)

dim_product["has_missing_metadata"] = (
    dim_product["productGroup"].isna()
    | dim_product["rrp"].isna()
)

dim_product["productGroup"] = (
    dim_product["productGroup"].astype("Int64")
)

In [22]:
dim_product.shape

(3822, 4)

In [23]:
dim_product["has_missing_metadata"].value_counts()

has_missing_metadata
False    3821
True        1
Name: count, dtype: int64

In [7]:
product_metrics = (
    fact_transactions.loc[
        ~fact_transactions["is_zero_value_item"]
    ]
    .groupby("articleID")
    .agg(
        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum"),
        ordered_sales_value=("ordered_sales_value", "sum"),
        returned_sales_value=("returned_sales_value", "sum"),
        kept_sales_value=("kept_sales_value", "sum")
    )
    .reset_index()
)

In [8]:
product_metrics["unit_return_rate"] = (
    product_metrics["returned_units"]
    / product_metrics["ordered_units"]
)

product_metrics["unit_retention_rate"] = (
    product_metrics["kept_units"]
    / product_metrics["ordered_units"]
)

In [9]:
dim_product = dim_product.merge(
    product_metrics,
    on="articleID",
    how="left"
)

In [11]:
dim_product.head()

,articleID,productGroup,rrp,ordered_units,returned_units,kept_units,ordered_sales_value,returned_sales_value,kept_sales_value,unit_return_rate,unit_retention_rate
0,i1000003,1.0,39.99,1.0,1.0,0.0,20.0,20.0,0.0,1.000000,0.000000
1,i1000005,1.0,39.99,1.0,1.0,0.0,10.0,10.0,0.0,1.000000,0.000000
2,i1000006,1.0,69.99,1.0,1.0,0.0,20.0,20.0,0.0,1.000000,0.000000
3,i1000007,1.0,69.99,2.0,1.0,1.0,40.0,20.0,20.0,0.500000,0.500000
4,i1000008,1.0,59.99,6.0,5.0,1.0,90.0,75.0,15.0,0.833333,0.166667


In [12]:
min_date = fact_transactions["orderDate"].min()
max_date = fact_transactions["orderDate"].max()

dim_date = pd.DataFrame({
    "date": pd.date_range(
        start=min_date,
        end=max_date,
        freq="D"
    )
})

In [13]:
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_of_week"] = dim_date["date"].dt.day_name()

In [14]:
dim_date.head()

,date,year,month,month_name,quarter,week,day,day_of_week
0,2014-01-01,2014,1,January,1,1,1,Wednesday
1,2014-01-02,2014,1,January,1,1,2,Thursday
2,2014-01-03,2014,1,January,1,1,3,Friday
3,2014-01-04,2014,1,January,1,1,4,Saturday
4,2014-01-05,2014,1,January,1,1,5,Sunday


In [15]:
product_attribute_check = (
    fact_transactions
    .groupby("articleID")
    .agg(
        product_group_values=(
            "productGroup",
            lambda x: x.nunique(dropna=True)
        ),
        rrp_values=(
            "rrp",
            lambda x: x.nunique(dropna=True)
        )
    )
)

product_attribute_check.head()

,product_group_values,rrp_values
articleID,,
i1000003,1,1
i1000005,1,1
i1000006,1,1
i1000007,1,1
i1000008,1,1


In [16]:
(product_attribute_check["product_group_values"] > 1).sum()

np.int64(0)

In [17]:
(product_attribute_check["rrp_values"] > 1).sum()

np.int64(0)

In [18]:
product_attribute_check["rrp_values"].value_counts().sort_index()

rrp_values
0       1
1    3821
Name: count, dtype: int64

In [19]:
dim_date.shape

(638, 8)

In [20]:
dim_date["date"].agg(["min", "max"])

min   2014-01-01
max   2015-09-30
Name: date, dtype: datetime64[us]

In [24]:
order_base = df.loc[
    df["is_valid_transaction"]
].copy()

order_consistency = (
    order_base
    .groupby("orderID")
    .agg(
        date_values=("orderDate", "nunique"),
        customer_values=("customerID", "nunique"),
        voucher_amount_values=("voucherAmount", "nunique"),
        voucher_id_values=(
            "voucherID",
            lambda x: x.nunique(dropna=True)
        )
    )
)

order_consistency.max()

date_values              1
customer_values          1
voucher_amount_values    1
voucher_id_values        1
dtype: int64

In [25]:
fact_orders = (
    order_base
    .groupby("orderID")
    .agg(
        orderDate=("orderDate", "first"),
        customerID=("customerID", "first"),

        voucherID=("voucherID", "first"),
        voucherAmount=("voucherAmount", "first"),
        has_voucher=("has_voucher", "max"),

        order_lines=("articleID", "size"),
        distinct_articles=("articleID", "nunique"),

        ordered_units=("ordered_units", "sum"),
        returned_units=("returned_units", "sum"),
        kept_units=("kept_units", "sum"),

        ordered_sales_value=("ordered_sales_value", "sum"),
        returned_sales_value=("returned_sales_value", "sum"),
        kept_sales_value=("kept_sales_value", "sum")
    )
    .reset_index()
)

In [26]:
fact_orders["has_missing_voucher_id"] = (
    fact_orders["has_voucher"]
    & fact_orders["voucherID"].isna()
)

In [27]:
fact_orders["has_missing_voucher_id"].value_counts()

has_missing_voucher_id
False    738697
True          1
Name: count, dtype: int64

In [28]:
fact_orders.shape

(738698, 15)

In [29]:
fact_orders.head()

,orderID,orderDate,customerID,voucherID,voucherAmount,has_voucher,order_lines,distinct_articles,ordered_units,returned_units,kept_units,ordered_sales_value,returned_sales_value,kept_sales_value,has_missing_voucher_id
0,a1000001,2014-01-01,c1010575,0,0.0,False,2,2,2,0,2,30.00,0.00,30.0,False
1,a1000002,2014-01-01,c1045905,0,0.0,False,2,2,2,1,1,84.99,49.99,35.0,False
2,a1000003,2014-01-01,c1089295,0,0.0,False,4,3,4,0,4,60.00,0.00,60.0,False
3,a1000004,2014-01-01,c1050116,0,0.0,False,1,1,1,1,0,89.99,89.99,0.0,False
4,a1000005,2014-01-01,c1089296,0,0.0,False,3,2,3,3,0,35.00,35.00,0.0,False


In [30]:
fact_orders["orderID"].is_unique

True

In [31]:
fact_orders[
    ["ordered_units", "returned_units", "kept_units"]
].sum()

ordered_units     2314801
returned_units    1212349
kept_units        1102452
dtype: int64

In [37]:
fact_transactions = fact_transactions.drop(
    columns=["voucherAmount"],
    errors="ignore"
)

In [33]:
model_validation = {
    "product_key_unique": dim_product["articleID"].is_unique,
    "date_key_unique": dim_date["date"].is_unique,
    "order_key_unique": fact_orders["orderID"].is_unique,

    "all_products_matched": (
        fact_transactions["articleID"]
        .isin(dim_product["articleID"])
        .all()
    ),

    "all_transaction_dates_matched": (
        fact_transactions["orderDate"]
        .isin(dim_date["date"])
        .all()
    ),

    "all_order_dates_matched": (
        fact_orders["orderDate"]
        .isin(dim_date["date"])
        .all()
    )
}

pd.Series(model_validation)

product_key_unique               True
date_key_unique                  True
order_key_unique                 True
all_products_matched             True
all_transaction_dates_matched    True
all_order_dates_matched          True
dtype: bool

In [34]:
unit_reconciliation = pd.DataFrame({
    "fact_transactions": fact_transactions[
        ["ordered_units", "returned_units", "kept_units"]
    ].sum(),

    "fact_orders": fact_orders[
        ["ordered_units", "returned_units", "kept_units"]
    ].sum()
})

unit_reconciliation

,fact_transactions,fact_orders
ordered_units,2314801,2314801
returned_units,1212349,1212349
kept_units,1102452,1102452


## Conclusiones del modelo BI

Este notebook transforma la capa analítica en un modelo dimensional preparado para su consumo en herramientas BI.

Se han definido dos granularidades principales:

- `fact_transactions`: una fila por línea de producto dentro de un pedido.
- `fact_orders`: una fila por pedido.

Además, se han construido `dim_product` y `dim_date` para separar los atributos descriptivos de las métricas transaccionales.

Las comprobaciones realizadas confirman que `productGroup` y `rrp` son atributos estables por `articleID`. La información de voucher, en cambio, pertenece al nivel pedido y se gestiona desde `fact_orders` para evitar duplicidades.

La reconciliación entre ambas tablas de hechos conserva exactamente:

- **2.314.801** unidades pedidas.
- **1.212.349** unidades devueltas.
- **1.102.452** unidades retenidas.

Esto confirma que el cambio de granularidad mantiene la consistencia de los principales totales analíticos.

### Resultado

El proyecto dispone ahora de una capa BI en la que cada atributo y métrica tiene una granularidad definida.

Esta estructura servirá como base para definir KPIs consistentes y construir posteriormente el modelo semántico y el dashboard en Power BI.